In [1]:
import os
import json
import io
import fitz  # PyMuPDF
from PIL import Image

## configure Gemini API key

In [2]:
import os
from getpass import getpass
from google import genai

# Prompt for the key if it's not already in the environment
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

# Initialize the Gemini client
client = genai.Client()
print("Google GenAI client configured successfully.")

Google GenAI client configured successfully.


In [3]:
# List all models available in the SDK
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

## uplod the pdf

In [4]:
pdf_path = "data/clg calendar 2022-2023 pdf.pdf"
doc = fitz.open(pdf_path)

print(f"Total Pages: {len(doc)}")

Total Pages: 22


# converting pdf to json 

In [5]:
from google import genai
from google.genai import types


# 1. Upload the PDF file directly via Files API
uploaded_pdf = client.files.upload(file=pdf_path)

# 2. Define Extraction Prompt
prompt = """
You are an expert OCR and data extraction system. 
Extract all information from all calendar pages in this PDF document.

For each row in every calendar table, extract:
- date: (e.g., "1", "2")
- day_of_week: (e.g., "THU", "FRI")
- month_year: (e.g., "September - 2022")
- event_description: (e.g., "Last Date for payment of ESE fee...", "Onam - Holiday", or "" if empty)
- cycle_number: (e.g., "6", or null if empty/holiday)
- day_order: (e.g., "III", "IV", "V", or null if empty/holiday)
- working_day_number: (e.g., "33", "34", or null if empty/holiday)
- is_holiday: (true/false)

Output strictly a JSON array containing all calendar row objects.
"""

# 3. Generate Content with Structured JSON Output
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=[uploaded_pdf, prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.1
    )
)

# 4. Save and Print JSON Data
extracted_data = response.text

with open("extracted_calendar.json", "w") as f:
    f.write(extracted_data)

print("Extraction completed and saved to extracted_calendar.json!\n")
print(extracted_data[:1000])  # Prints first 1000 characters preview

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Extraction completed and saved to extracted_calendar.json!

[
  {
    "date": "1",
    "day_of_week": "FRI",
    "month_year": "July - 2022",
    "event_description": "",
    "cycle_number": null,
    "day_order": null,
    "working_day_number": null,
    "is_holiday": true
  },
  {
    "date": "2",
    "day_of_week": "SAT",
    "month_year": "July - 2022",
    "event_description": "",
    "cycle_number": null,
    "day_order": null,
    "working_day_number": null,
    "is_holiday": true
  },
  {
    "date": "3",
    "day_of_week": "SUN",
    "month_year": "July - 2022",
    "event_description": "",
    "cycle_number": null,
    "day_order": null,
    "working_day_number": null,
    "is_holiday": true
  },
  {
    "date": "4",
    "day_of_week": "MON",
    "month_year": "July - 2022",
    "event_description": "",
    "cycle_number": null,
    "day_order": null,
    "working_day_number": null,
    "is_holiday": true
  },
  {
    "date": "5",
    "day_of_week": "TUE",
    "month_year": "

# chunking

In [6]:
import json

def build_rag_chunks(json_file_path):
    # Load the JSON data generated in Step 1
    with open(json_file_path, "r") as f:
        calendar_data = json.load(f)
    
    chunks = []
    metadata_list = []
    
    for idx, entry in enumerate(calendar_data):
        date = entry.get("date", "")
        day_of_week = entry.get("day_of_week", "")
        month_year = entry.get("month_year", "")
        event = entry.get("event_description", "")
        cycle = entry.get("cycle_number")
        day_order = entry.get("day_order")
        working_day = entry.get("working_day_number")
        is_holiday = entry.get("is_holiday", False)

        # 1. Construct Natural Language String (The Document Chunk)
        chunk_parts = [f"Date: {date} {day_of_week}, {month_year}."]
        
        if is_holiday:
            chunk_parts.append("Status: Holiday.")
        else:
            chunk_parts.append("Status: Working Day.")
            
        if event:
            chunk_parts.append(f"Event Details: {event}.")
        if cycle:
            chunk_parts.append(f"Cycle Number: {cycle}.")
        if day_order:
            chunk_parts.append(f"Day Order: {day_order}.")
        if working_day:
            chunk_parts.append(f"Working Day Number: {working_day}.")

        # Join into a single searchable string
        chunk_text = " ".join(chunk_parts)
        chunks.append(chunk_text)
        
        # 2. Extract Metadata (For structured filtering later if needed)
        metadata = {
            "date": str(date),
            "day_of_week": str(day_of_week),
            "month_year": str(month_year),
            "day_order": str(day_order) if day_order else "None",
            "is_holiday": is_holiday
        }
        metadata_list.append(metadata)

    return chunks, metadata_list

# Execute Chunking Function
chunks, metadatas = build_rag_chunks("extracted_calendar.json")

# Inspect the first 3 chunks
print(f"Total Chunks Created: {len(chunks)}\n")
print("--- Sample Chunks ---")
for c in chunks[:3]:
    print(c)

Total Chunks Created: 335

--- Sample Chunks ---
Date: 1 FRI, July - 2022. Status: Holiday.
Date: 2 SAT, July - 2022. Status: Holiday.
Date: 3 SUN, July - 2022. Status: Holiday.


# Embedding and storing in to vector db

In [7]:
import chromadb

# 1. Initialize ChromaDB Persistent Client
chroma_client = chromadb.PersistentClient(path="./calendar_vector_db")

# Delete old collection if it exists to ensure a clean start
try:
    chroma_client.delete_collection(name="college_calendar")
except Exception:
    pass

# Create a new collection using ChromaDB's default embedding function
collection = chroma_client.create_collection(name="college_calendar")

# Generate document IDs
ids = [f"doc_{i}" for i in range(len(chunks))]

# 2. Add Documents to ChromaDB
# ChromaDB handles local embedding and vector generation automatically!
print("Embedding and indexing chunks into ChromaDB locally...")

collection.add(
    documents=chunks,
    metadatas=metadatas,
    ids=ids
)

print(f"\nSuccessfully embedded and stored all {len(chunks)} records into ChromaDB!")

Embedding and indexing chunks into ChromaDB locally...

Successfully embedded and stored all 335 records into ChromaDB!


# RAG Retrieval & LLM Answer Generation Engine

In [8]:
# Step 4: Accurate Month-Specific RAG Pipeline (Python Native Filtering)

def ask_calendar(user_query):
    # 1. Extract month name from user query
    months = ["january", "february", "march", "april", "may", "june", 
              "july", "august", "september", "october", "november", "december"]
    
    found_month = None
    for m in months:
        if m in user_query.lower():
            found_month = m.capitalize()
            break

    # 2. Fetch ALL documents from ChromaDB collection
    all_records = collection.get()
    all_chunks = all_records['documents']

    # 3. Filter documents matching ONLY the queried month (e.g., "August")
    if found_month:
        # Strict match for the requested month inside the chunk text/JSON
        month_chunks = [
            chunk for chunk in all_chunks 
            if f'"month_year": "{found_month}' in chunk or f'month_year: {found_month}' in chunk or found_month in chunk
        ]
    else:
        month_chunks = all_chunks

    # 4. Filter ONLY 'is_holiday: true' chunks for that specific month
    holiday_chunks = [
        chunk for chunk in month_chunks 
        if "'is_holiday': true" in chunk.lower() 
        or '"is_holiday": true' in chunk.lower() 
        or "is_holiday: true" in chunk.lower()
    ]

    # Combine context (strictly August holidays only)
    final_context = "\n\n".join(holiday_chunks) if holiday_chunks else "\n\n".join(month_chunks)

    # 5. Strict Prompt for Gemini LLM
    prompt = f"""
    You are an intelligent College Calendar Assistant.
    You ONLY have data for the 2022-2023 Academic Year.

    CRITICAL RULES:
    1. Check if the user is asking about a specific year (e.g., 2024, 2025, 2026).
    2. If the user asks about ANY year other than 2022 or 2023, IMMEDIATELY state: "Information Not Available in Calendar Context for the requested year."
    3. Do NOT substitute 2022 or 2023 data if the user asks for another year (like 2025).
    4. If the question is valid for 2022-2023, list the exact details from the context below.

    Calendar Context:
    {final_context}

    User Question: {user_query}

    Answer:
    """

    # 6. Generate Response using Gemini Model
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

# Test the RAG Pipeline
question = "How many holidays in August?"
print(f"Question: {question}\n")
answer = ask_calendar(question)
print(f"Answer:\n{answer}")

Question: How many holidays in August?

Answer:
Based on the 2022 calendar context, there are **11 holidays** in August 2022:

1. **August 3 (WED)** - Status: Holiday | Event Details: Adi Peruku - Compensatory Holiday
2. **August 6 (SAT)** - Status: Holiday
3. **August 7 (SUN)** - Status: Holiday
4. **August 9 (TUE)** - Status: Holiday | Event Details: Muharram - Holiday
5. **August 14 (SUN)** - Status: Holiday
6. **August 15 (MON)** - Status: Holiday | Event Details: Independence Day - Holiday
7. **August 19 (FRI)** - Status: Holiday | Event Details: Krishna Jayanthi - Holiday
8. **August 20 (SAT)** - Status: Holiday
9. **August 21 (SUN)** - Status: Holiday
10. **August 28 (SUN)** - Status: Holiday
11. **August 31 (WED)** - Status: Holiday | Event Details: Vinayakar Chathurthi - Holiday
